# 01 Pipeline Smoke + Preview

Dieses Notebook unterstützt das erste Laden der Rohdaten:
- SEC User Agent laden
- MVP Pipeline bauen
- Outputs speichern
- Erste Vorschau der Daten

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from data_pipeline.run_mvp import resolve_sec_user_agent

sec_user_agent = resolve_sec_user_agent(PROJECT_ROOT)
if not sec_user_agent:
    raise RuntimeError('SEC_USER_AGENT fehlt. Bitte .env aus .env.example anlegen.')

sec_user_agent

In [ ]:
from data_pipeline.api import build_dataset

symbols = ['AAPL', 'MSFT', 'NVDA']
start = '2022-01-01'
end = '2024-12-31'

result = build_dataset(symbols=symbols, start=start, end=end, sec_user_agent=sec_user_agent)
len(result.prices), len(result.fundamentals), len(result.features)

In [ ]:
out_dir = PROJECT_ROOT / 'artifacts' / 'real_data_mvp'
out_dir.mkdir(parents=True, exist_ok=True)

prices_path = out_dir / 'prices_daily.csv'
fund_path = out_dir / 'fundamentals_quarterly.csv'
feat_path = out_dir / 'features_daily.csv'

result.prices.to_csv(prices_path, index=False)
result.fundamentals.to_csv(fund_path, index=False)
result.features.to_csv(feat_path, index=False)

prices_path, fund_path, feat_path

In [ ]:
print("prices.head()")
display(result.prices.head())

print("fundamentals.head()")
display(result.fundamentals.head())

print("features.head()")
display(result.features.head())

## Erweiterte Datenanalyse im Smoke-Notebook

Diese Sektion gibt dir direkt im ersten Notebook dieselben wichtigen Checks wie im Analyse-Notebook:\n- Missing Values\n- QA und Zeitlogik\n- Event-Effekte um Reports\n- Fundamentaldaten-Plausibilität\n- einfacher Signal-Sanity-Check

In [ ]:
import numpy as np
import pandas as pd

prices = result.prices.copy()
fundamentals = result.fundamentals.copy()
features = result.features.copy()

def missingness_table(df, dataset_name):
    miss = df.isna().mean().sort_values(ascending=False).rename('missing_ratio')
    out = miss.to_frame()
    out['dataset'] = dataset_name
    out['dtype'] = [str(df[c].dtype) for c in out.index]
    return out.reset_index(names='column')

missing_prices = missingness_table(prices, 'prices')
missing_fund = missingness_table(fundamentals, 'fundamentals')
missing_feat = missingness_table(features, 'features')

print('Top Missingness - prices')
display(missing_prices.head(10))

print('Top Missingness - fundamentals')
display(missing_fund.head(15))

print('Top Missingness - features')
display(missing_feat.head(15))

In [ ]:
# QA: Duplikate, Coverage, Filing-Lag-Verteilung
dup_prices = int(prices.duplicated(subset=['symbol', 'date']).sum())
dup_fund = int(fundamentals.duplicated(subset=['symbol', 'period_end', 'report_date', 'report_type']).sum())
negative_lag_rows = int((fundamentals['filing_lag_days'] < 0).sum())

coverage_by_symbol = (
    prices.groupby('symbol', as_index=False)
    .agg(n_days=('date', 'size'), start=('date', 'min'), end=('date', 'max'))
    .sort_values('symbol')
)

lag_stats = fundamentals['filing_lag_days'].describe(percentiles=[0.1, 0.5, 0.9, 0.95]).to_frame(name='filing_lag_days')

print({'duplicate_prices_rows': dup_prices, 'duplicate_fund_rows': dup_fund, 'negative_filing_lag_rows': negative_lag_rows})
print('Price coverage by symbol')
display(coverage_by_symbol)
print('Filing lag distribution')
display(lag_stats)

In [ ]:
# Event-Analyse: Report-Tage vs normale Tage
feat_evt = features.copy().sort_values(['symbol', 'date']).reset_index(drop=True)
feat_evt['ret_1d'] = feat_evt.groupby('symbol')['close'].pct_change()

event_stats = (
    feat_evt.groupby('is_report_day')['ret_1d']
    .agg(['count', 'mean', 'std', 'median'])
    .rename(index={0: 'non_report_day', 1: 'report_day'})
)

event_window = feat_evt[feat_evt['days_since_report'].between(0, 3, inclusive='both')].copy()
non_event_window = feat_evt[feat_evt['days_since_report'] > 20].copy()

window_compare = {
    'event_window_rows': int(len(event_window)),
    'non_event_window_rows': int(len(non_event_window)),
    'event_window_abs_ret_mean': float(event_window['ret_1d'].abs().mean()),
    'non_event_window_abs_ret_mean': float(non_event_window['ret_1d'].abs().mean()),
}

display(event_stats)
print(window_compare)

In [ ]:
# Fundamentals-Plausibilität: einfache Bereichschecks
plaus = {
    'gross_margin_outside_[0,1]': int(((features['gross_margin'] < 0) | (features['gross_margin'] > 1)).sum()),
    'roe_outside_[-2,2]': int(((features['roe'] < -2) | (features['roe'] > 2)).sum()),
    'debt_to_equity_gt_20': int((features['debt_to_equity'] > 20).sum()),
    'revenue_non_positive_with_fundamentals': int(((features['has_fundamentals'] == 1) & (features['revenue'] <= 0)).sum()),
}

quantiles = features[[
    'revenue',
    'net_income',
    'operating_cashflow',
    'debt_to_equity',
    'gross_margin',
    'roe',
]].quantile([0.01, 0.05, 0.5, 0.95, 0.99])

print(plaus)
display(quantiles)

In [ ]:
# Einfacher Signal-Sanity-Check (keine Modellierung, nur Gefühl für lineare Zusammenhänge)
signal = features.copy().sort_values(['symbol', 'date']).reset_index(drop=True)
signal['fwd_ret_5d'] = signal.groupby('symbol')['close'].pct_change(5).shift(-5)

valid = signal[signal['has_fundamentals'] == 1].copy()
valid = valid.dropna(subset=['fwd_ret_5d', 'roe', 'gross_margin', 'debt_to_equity'])

corrs = {
    'corr_roe_vs_fwd_ret_5d': float(valid['roe'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'corr_gross_margin_vs_fwd_ret_5d': float(valid['gross_margin'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'corr_debt_to_equity_vs_fwd_ret_5d': float(valid['debt_to_equity'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'n_valid_rows': int(len(valid)),
}

print(corrs)